# DA6401 — Remap + Continue Training from Friend's Checkpoints

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Save & Run All (Commit)**

Friend uses different key names — this notebook remaps them to our architecture then trains 5 more epochs.

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available()

In [ ]:
import gdown

CLASSIFIER_ID = "1Ob4ecaMOaoH-56TdIQ9lYEdGKfn1jy8L"
LOCALIZER_ID  = "1jRM83Xr6HETheUhv1MQqRMEl2eAwbkSK"
UNET_ID       = "1u-ux_IFhPResLG7TPl8u0qcJkcAvKLZL"

for drive_id, out_path, label in [
    (CLASSIFIER_ID, f"{CKPT}/classifier_friend.pth", "classifier"),
    (LOCALIZER_ID,  f"{CKPT}/localizer_friend.pth",  "localizer"),
    (UNET_ID,       f"{CKPT}/unet_friend.pth",        "unet"),
]:
    print(f"Downloading {label}...")
    gdown.download(id=drive_id, output=out_path, quiet=True, fuzzy=True)
    mb = os.path.getsize(out_path) / 1e6
    print(f"  ✓ {mb:.0f} MB")

In [ ]:
import re, torch

# ── Key remapping: friend's naming → our naming ──────────────
#
# Friend encoder: encoder.blockN.M.L.suffix
#   N = block number (1-5),  M = sub-block (0 or 1),  L = layer (0=Conv, 1=BN)
# Our encoder:    encoder.blockX.Y.suffix
#   X = block number (0-4),  Y = flat index in Sequential
#
# Block mapping:  friend block1→ours block0, ..., friend block5→ours block4
# Single-conv blocks (0,1): M always 0,  Conv→Y=0, BN→Y=1
# Double-conv blocks (2,3,4): M=0→Y=0/1, M=1→Y=3/4  (skip Y=2=ReLU)

def remap_encoder(key):
    m = re.match(r'encoder\.block(\d+)\.(\d+)\.(\d+)\.(.*)', key)
    if not m:
        return None
    N, M, L, suffix = int(m.group(1)), int(m.group(2)), int(m.group(3)), m.group(4)
    X = N - 1  # shift: 1→0, 2→1, ..., 5→4
    if X in [0, 1]:           # single-conv block
        Y = 0 if L == 0 else 1
    else:                      # double-conv block
        if M == 0:   Y = 0 if L == 0 else 1
        else:        Y = 3 if L == 0 else 4
    return f'encoder.block{X}.{Y}.{suffix}'


def remap_bottleneck(key):
    # Friend: no CustomDropout → indices 0,1,2(ReLU),3(Conv2),4(BN2)
    # Ours:   CustomDropout at 3 → indices 0,1,2(ReLU),3(Drop),4(Conv2),5(BN2)
    m = re.match(r'bottleneck\.(\d+)\.(.*)', key)
    if not m:
        return None
    idx, suffix = int(m.group(1)), m.group(2)
    shift = {0: 0, 1: 1, 3: 4, 4: 5}  # skip our index 3 (CustomDropout)
    if idx in shift:
        return f'bottleneck.{shift[idx]}.{suffix}'
    return None


def remap_unet_decoder(key):
    # upN.weight/bias  →  up(N-1).up.weight/bias
    # decN.X.suffix    →  up(N-1).conv.X.suffix
    # head.weight/bias →  final_conv.weight/bias
    if key in ('head.weight', 'head.bias'):
        return 'final_conv.' + key[5:]
    m = re.match(r'up(\d+)\.(weight|bias)', key)
    if m:
        our_n = int(m.group(1)) - 1
        return f'up{our_n}.up.{m.group(2)}'
    m = re.match(r'dec(\d+)\.(\d+)\.(.*)', key)
    if m:
        our_n = int(m.group(1)) - 1
        return f'up{our_n}.conv.{m.group(2)}.{m.group(3)}'
    return None


def remap_state_dict(sd, model_type):
    """Convert friend's state dict to our key convention."""
    new_sd = {}
    skipped = []
    for k, v in sd.items():
        new_k = None
        if k.startswith('encoder.'):
            new_k = remap_encoder(k)
        elif k.startswith('bottleneck.'):
            new_k = remap_bottleneck(k)
        elif k.startswith('classifier.'):
            new_k = 'head.' + k[len('classifier.'):]  # classifier→head
        elif k.startswith('regressor.'):
            new_k = 'head.' + k[len('regressor.'):]   # regressor→head
        elif model_type == 'unet':
            new_k = remap_unet_decoder(k)
        if new_k:
            new_sd[new_k] = v
        else:
            skipped.append(k)
    if skipped:
        print(f"  Skipped {len(skipped)} keys: {skipped[:5]}")
    return new_sd


# Apply remapping and save
for src, dst, mtype in [
    (f"{CKPT}/classifier_friend.pth", f"{CKPT}/classifier.pth", 'classifier'),
    (f"{CKPT}/localizer_friend.pth",  f"{CKPT}/localizer.pth",  'localizer'),
    (f"{CKPT}/unet_friend.pth",        f"{CKPT}/unet.pth",        'unet'),
]:
    sd = torch.load(src, map_location='cpu', weights_only=False)
    new_sd = remap_state_dict(sd, mtype)
    torch.save(new_sd, dst)
    print(f"  {mtype}: {len(sd)} keys → {len(new_sd)} remapped keys")

In [ ]:
from data.pets_dataset import download_oxford_pet, create_dataloaders
download_oxford_pet("./data/oxford_pet")
train_loader, val_loader, _ = create_dataloaders(
    root="./data/oxford_pet", batch_size=32, num_workers=2
)
print("Dataset ready.")

## 1. Baseline — friend's weights after remapping

In [ ]:
from sklearn.metrics import f1_score
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from train import compute_iou, compute_dice_score

device = torch.device("cuda")

print("="*55)
print("  FRIEND'S WEIGHTS — BASELINE (after remap)")
print("="*55)

# Classification
try:
    m = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
    missing, unexpected = m.load_state_dict(
        torch.load(f"{CKPT}/classifier.pth", map_location=device, weights_only=False),
        strict=False)
    if missing: print(f"  [cls] Missing: {missing[:3]}")
    m.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in val_loader:
            preds.extend(m(b["image"].to(device)).argmax(1).cpu().tolist())
            labels.extend(b["label"].tolist())
    f1 = f1_score(labels, preds, average="macro")
    print(f"  Classification  F1  = {f1:.4f}  (need ≥ 0.80)  {'✓' if f1>=0.80 else '✗'}")
except Exception as e:
    print(f"  Classification  ERROR: {e}")

# Localization
try:
    m = VGG11Localizer().to(device)
    missing, _ = m.load_state_dict(
        torch.load(f"{CKPT}/localizer.pth", map_location=device, weights_only=False),
        strict=False)
    if missing: print(f"  [loc] Missing: {missing[:3]}")
    m.eval()
    iou_sum, iou50, total = 0.0, 0, 0
    with torch.no_grad():
        for b in val_loader:
            ious = compute_iou(m(b["image"].to(device)).cpu(), b["bbox"])
            iou_sum += ious.sum().item(); iou50 += (ious>=0.5).sum().item(); total += len(ious)
    print(f"  Localization    IoU = {iou_sum/total:.4f},  Acc@0.5 = {iou50/total*100:.1f}%  (need ≥ 60%)  {'✓' if iou50/total>=0.6 else '✗'}")
except Exception as e:
    print(f"  Localization    ERROR: {e}")

# Segmentation
try:
    m = VGG11UNet(num_classes=3).to(device)
    missing, _ = m.load_state_dict(
        torch.load(f"{CKPT}/unet.pth", map_location=device, weights_only=False),
        strict=False)
    if missing: print(f"  [seg] Missing: {missing[:3]}")
    m.eval()
    dice_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = m(b["image"].to(device)).argmax(1).cpu()
            dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
    print(f"  Segmentation    Dice = {dice_sum/total:.4f}  (need ≥ 0.30)  {'✓' if dice_sum/total>=0.30 else '✗'}")
except Exception as e:
    print(f"  Segmentation    ERROR: {e}")

print("="*55)

## 2. Continue Localizer (+5 epochs)

In [ ]:
import time, torch.nn as nn, wandb
from losses.iou_loss import IoULoss

model = VGG11Localizer().to(device)
model.load_state_dict(
    torch.load(f"{CKPT}/localizer.pth", map_location=device, weights_only=False),
    strict=False)

model.eval()
iou_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
        iou_sum += ious.sum().item(); total += len(ious)
baseline_iou = iou_sum / total
print(f"Localizer baseline: {baseline_iou:.4f}")

wandb.init(project="da6401-assignment2", name="localizer_friends_remap_5ep",
           tags=["continue", "localization"])

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=1e-6)
mse_fn, iou_fn = nn.MSELoss(), IoULoss(reduction="mean")

best_iou = baseline_iou
for epoch in range(1, 6):
    model.train(); t0 = time.time()
    for b in train_loader:
        imgs, bboxes = b["image"].to(device), b["bbox"].to(device)
        optimizer.zero_grad()
        pred = model(imgs)
        (mse_fn(pred, bboxes) + iou_fn(pred, bboxes)).backward()
        optimizer.step()
    scheduler.step()
    model.eval(); iou_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
            iou_sum += ious.sum().item(); total += len(ious)
    val_iou = iou_sum / total
    wandb.log({"val/iou": val_iou})
    print(f"  Epoch {epoch}/5 | Val IoU: {val_iou:.4f} | {time.time()-t0:.0f}s")
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), f"{CKPT}/localizer.pth")
        print(f"    ✓ Saved!")

wandb.finish()
print(f"\nLocalizer: {baseline_iou:.4f} → {best_iou:.4f}")

## 3. Continue U-Net (+5 epochs, partial freeze)

In [ ]:
import gdown

# Friend's UNet decoder uses half-sized channels — incompatible shape, can't load.
# Instead, download OUR own best unet.pth (partial freeze, val Dice ≈ 0.82) from Drive.
OUR_UNET_DRIVE_ID = "1527bugKX9NKd_JAOltXdH_rpU1j6Spzj"
unet_path = f"{CKPT}/unet.pth"
print("Downloading our unet.pth from Drive (partial-freeze, Dice≈0.82)...")
gdown.download(id=OUR_UNET_DRIVE_ID, output=unet_path, quiet=False, fuzzy=True)
print(f"✓ {os.path.getsize(unet_path)/1e6:.0f} MB")

seg_model = VGG11UNet(num_classes=3).to(device)
seg_model.load_state_dict(
    torch.load(unet_path, map_location=device, weights_only=False),
    strict=False)

# Partial freeze: blocks 0-2 frozen, blocks 3-4 + decoder trainable
for name, param in seg_model.encoder.named_parameters():
    param.requires_grad = not any(name.startswith(b) for b in
                                  ["block0","block1","block2","pool0","pool1","pool2"])

seg_model.eval(); dice_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        pred = seg_model(b["image"].to(device)).argmax(1).cpu()
        dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
baseline_dice = dice_sum / total
print(f"U-Net baseline Dice: {baseline_dice:.4f}")

wandb.init(project="da6401-assignment2", name="unet_our_weights_5ep",
           tags=["continue", "segmentation"])

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.0, 2.0]).to(device))
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, seg_model.parameters()), lr=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=1e-6)

best_dice = baseline_dice
for epoch in range(1, 6):
    seg_model.train(); t0 = time.time()
    for b in train_loader:
        imgs, masks = b["image"].to(device), b["mask"].to(device)
        optimizer.zero_grad()
        criterion(seg_model(imgs), masks).backward()
        optimizer.step()
    scheduler.step()
    seg_model.eval(); dice_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = seg_model(b["image"].to(device)).argmax(1).cpu()
            dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
    val_dice = dice_sum / total
    wandb.log({"val/dice": val_dice})
    print(f"  Epoch {epoch}/5 | Val Dice: {val_dice:.4f} | {time.time()-t0:.0f}s")
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(seg_model.state_dict(), unet_path)
        print(f"    ✓ Saved!")

wandb.finish()
print(f"\nU-Net: {baseline_dice:.4f} → {best_dice:.4f}")

In [ ]:
print("="*55)
print("  FILES — download from Output tab")
print("="*55)
for f in sorted(os.listdir(CKPT)):
    if not f.endswith('_friend.pth'):  # skip originals
        mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
        print(f"  {f:30s}  {mb:.0f} MB")
print("\n  → Upload to Drive, paste new IDs to update multitask.py")